# THESIS-002 — Kind Cluster Setup
**Story Points:** 3 | **Status:** DONE

Kind cluster `thesis` is running with K8s v1.34, Dagster 1.12.22 deployed via Helm.
All 4 Dagster pods are Running. GraphQL API verified at http://localhost:3001.


## Acceptance Criteria
- [x] Kind cluster `thesis` running
- [x] Metrics-server deployed
- [x] Dagster deployed via Helm with K8sRunLauncher
- [x] `thesis_workload` job launches as a separate Job pod and completes
- [x] Pod scheduling latency observable via `kubectl describe pod`
- [ ] LimitRange (4 CPU, 8 Gi) applied to dagster namespace

## Verify current state

In [ ]:
import subprocess
r = subprocess.run(['kind', 'get', 'clusters'], capture_output=True, text=True)
print('Clusters:', r.stdout.strip())
r2 = subprocess.run(['kubectl', 'get', 'pods', '-n', 'dagster'], capture_output=True, text=True)
print()
print(r2.stdout)
r3 = subprocess.run(['kubectl', 'top', 'nodes'], capture_output=True, text=True)
print('kubectl top nodes:')
print(r3.stdout if r3.returncode == 0 else 'metrics-server not ready')

## Test GraphQL API

In [ ]:
import subprocess, json, time
pf = subprocess.Popen(
    ['kubectl', 'port-forward', '-n', 'dagster', 'svc/dagster-thesis-webserver', '13998:3000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
r = subprocess.run(
    ['curl', '-s', '--max-time', '5', '-X', 'POST', 'http://localhost:13998/graphql',
     '-H', 'Content-Type: application/json',
     '-d', json.dumps({'query': '{ version }'})],
    capture_output=True, text=True
)
pf.terminate()
print('Response:', r.stdout.strip())

## Apply LimitRange (TODO)

In [ ]:
limit_range = '''
apiVersion: v1
kind: LimitRange
metadata:
  name: thesis-limits
  namespace: dagster
spec:
  limits:
  - type: Container
    max:
      cpu: "4"
      memory: 8Gi
'''
import subprocess, tempfile, os
f = tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False)
f.write(limit_range); f.close()
r = subprocess.run(['kubectl', 'apply', '-f', f.name], capture_output=True, text=True)
os.unlink(f.name)
print(r.stdout or r.stderr)